# Notebook 04: ArcGIS Web Maps and Dashboard

**Publishing the Lake Mead near-shore clustering results and building an interactive comparison dashboard**

Notebook 03 produced three CSVs in `data/processed/`:

- `mead_fires_clustered.csv`: all 844 wildfires with cluster labels from each method (basin and near-shore)
- `mead_cluster_summary.csv`: per-cluster summary stats for the near-shore subset (8 rows: 4 per method)
- `mead_global_metrics.csv`: headline metrics for the near-shore subset (single row)

This notebook focuses on the **near-shore subset** (214 fires within 5 km of the lake) because that's where the clustering methods differ most visibly: obstacle-aware $k$-means with optimized $\beta$ produces about a 19% arc-length span improvement over standard $k$-means on this subset, and the smaller fire count makes individual cluster shifts easier to see.

The notebook uploads the near-shore data to ArcGIS Online, publishes it as a hosted feature layer, and walks through building two web maps (one styled by standard $k$-means clusters, one by obstacle-aware clusters) and a dashboard that puts them side by side.

## 1. Setup

In [1]:
# Standard packages
import getpass
from pathlib import Path

import numpy as np
import pandas as pd
import shutil
import geopandas as gpd
from shapely.geometry import Polygon

# ArcGIS Python API
from arcgis.gis import GIS

# Cluster colors from the project palette (reference for the Map Viewer step)
CLUSTER_COLORS = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
print('Cluster colors for the Map Viewer:')
for i, c in enumerate(CLUSTER_COLORS, start=1):
    print(f'  Cluster {i}: {c}')

Cluster colors for the Map Viewer:
  Cluster 1: #e74c3c
  Cluster 2: #3498db
  Cluster 3: #2ecc71
  Cluster 4: #f39c12


## 2. Loading Results from Notebook 03

Three CSVs feed the dashboard. The fires CSV holds all 844 fires (basin-wide) along with cluster labels; we'll filter it to the near-shore subset before publishing. The cluster summary and global metrics CSVs were saved with near-shore values directly in Notebook 03 and don't need filtering here.

In [2]:
processed_dir = Path('../data/processed')

fires = pd.read_csv(processed_dir / 'mead_fires_clustered.csv')
cluster_summary = pd.read_csv(processed_dir / 'mead_cluster_summary.csv')
global_metrics = pd.read_csv(processed_dir / 'mead_global_metrics.csv')

print(f'Loaded {len(fires)} fires (full basin)')
print(f'Cluster summary rows: {len(cluster_summary)} (4 std + 4 oa_opt for near-shore)')
print(f'Global metrics rows: {len(global_metrics)}')
print()
print('Cluster columns available in fires:')
for col in fires.columns:
    if 'cluster' in col:
        print(f'  {col}')

Loaded 844 fires (full basin)
Cluster summary rows: 8 (4 std + 4 oa_opt for near-shore)
Global metrics rows: 1

Cluster columns available in fires:
  cluster_basin_opt
  cluster_basin_std
  cluster_basin_eq
  cluster_near_std
  cluster_near_eq
  cluster_near_opt


In [3]:
# Headline numbers for the dashboard indicators
global_metrics.round(4)

,lake,subset,threshold_km,total_fires,year_min,year_max,k,opt_beta,span_std,span_oa_opt,span_improvement_pct
0,Mead,near_shore,5,214,1992,2020,4,1.08,0.2425,0.1964,19.0135


In [4]:
# Per-cluster numbers for the dashboard side panels
cluster_summary.round(2)

,method,cluster_id,cluster_label,n_fires,mean_fire_size,median_fire_size,pct_human,pct_natural,cluster_span
0,std,0,1,32,1.75,0.10,65.62,34.38,0.42
1,std,1,2,40,11.00,0.15,75.00,25.00,0.19
2,std,2,3,16,60.39,0.10,25.00,75.00,0.16
3,std,3,4,126,0.26,0.10,75.40,24.60,0.21
4,oa_opt,0,1,34,23.94,0.10,47.06,52.94,0.24
5,oa_opt,1,2,40,11.00,0.15,75.00,25.00,0.19
6,oa_opt,2,3,14,14.87,0.30,64.29,35.71,0.16
7,oa_opt,3,4,126,0.26,0.10,75.40,24.60,0.21


## 3. Preparing Data for ArcGIS

The CSV that gets uploaded should contain only the near-shore fires (those within 5 km of Lake Mead's shore) with the near-shore cluster labels. We:

- Filter `fires` to the near-shore subset using the `dist_to_lake_km` column
- Use `cluster_near_std` and `cluster_near_opt` as the source for the published `Cluster_Std` and `Cluster_OA` columns
- Add friendly cause and fire size labels
- Cast year to a string so ArcGIS doesn't render it with comma separators

In [5]:
# Filter to near-shore fires
threshold_km = 5
near_mask = fires['dist_to_lake_km'] < threshold_km
fires_pub = fires[near_mask].copy().reset_index(drop=True)

print(f'Near-shore subset: {len(fires_pub)} fires within {threshold_km} km of Lake Mead')

# 1-indexed cluster labels (the dashboard displays Cluster 1, 2, 3, 4)
fires_pub['Cluster_Std'] = (fires_pub['cluster_near_std'] + 1).astype(int)
fires_pub['Cluster_OA'] = (fires_pub['cluster_near_opt'] + 1).astype(int)

# Cause labels
fires_pub['Cause'] = fires_pub['cause_binary'].map({0: 'Natural', 1: 'Human-Caused'})
fires_pub['Specific_Cause'] = fires_pub['NWCG_GENERAL_CAUSE'].replace('Natural', 'Lightning')

# Round fire size for popup readability
fires_pub['Fire_Size_Acres'] = fires_pub['FIRE_SIZE'].round(2)

# Cast year to string so ArcGIS doesn't render it as "2,003"
fires_pub['Fire_Year'] = fires_pub['FIRE_YEAR'].astype(int).astype(str)

# Distance to lake (handy for popup readers who want context)
fires_pub['Dist_To_Lake_Km'] = fires_pub['dist_to_lake_km'].round(2)

# Columns to publish
publish_cols = [
    'LATITUDE', 'LONGITUDE',
    'Cluster_Std',
    'Cluster_OA',
    'Cause', 'Specific_Cause',
    'Fire_Size_Acres', 'Fire_Year',
    'Dist_To_Lake_Km',
]
fires_pub = fires_pub[publish_cols].copy()

# Save the publishing-ready CSV
publish_path = processed_dir / 'mead_fires_nearshore_webmap.csv'
fires_pub.to_csv(publish_path, index=False)

print(f'\nSaved {len(fires_pub)} near-shore fires to {publish_path.name}')
print()
print('Column dtypes:')
print(fires_pub.dtypes)
print()
print('Standard k-means cluster distribution:')
print(fires_pub['Cluster_Std'].value_counts().sort_index().to_string())
print()
print('OA optimized cluster distribution:')
print(fires_pub['Cluster_OA'].value_counts().sort_index().to_string())

Near-shore subset: 214 fires within 5 km of Lake Mead

Saved 214 near-shore fires to mead_fires_nearshore_webmap.csv

Column dtypes:
LATITUDE           float64
LONGITUDE          float64
Cluster_Std          int64
Cluster_OA           int64
Cause               object
Specific_Cause      object
Fire_Size_Acres    float64
Fire_Year           object
Dist_To_Lake_Km    float64
dtype: object

Standard k-means cluster distribution:
Cluster_Std
1     32
2     40
3     16
4    126

OA optimized cluster distribution:
Cluster_OA
1     34
2     40
3     14
4    126


## 4. Connecting to ArcGIS Online

The script uploads to your AGOL organization using the ArcGIS Python API. Authentication uses a username and password prompt; the password isn't stored anywhere.

In [6]:
gis = GIS(
    url='https://michele-75.maps.arcgis.com',
    username='Michele-75',
    password=getpass.getpass('Enter your ArcGIS Online password: '),
)

print(f'Connected as: {gis.users.me.username}')
print(f'Organization: {gis.properties.name}')

Connected as: Michele-75
Organization: Michele Perry


## 5. Publishing the Hosted Items

Four items get uploaded:

1. **Near-shore fires feature layer** -- 214 points, one per fire, with cluster columns for both methods.
2. **Cluster summary table** -- 8 rows (4 per method) with per-cluster stats for the dashboard side panels.
3. **Global metrics table** -- 1 row with the headline numbers (including the 19% span improvement).
4. **Lake Mead boundary feature layer** -- the simplified shoreline polygon, uploaded as a hosted layer so it can be added to both web maps for visual reference.

By default the script does NOT delete existing items with the same titles. If you've published these before, the new uploads will be created alongside (with " (1)", " (2)" appended). To delete and recreate cleanly, change `delete_existing` to `True`.

If you'd rather update existing items in place (preserves item IDs and any dashboard configurations that reference them), use the Update Data flow at the bottom of this section.

In [7]:
delete_existing = False

# Titles we'll use for the items we're about to create
item_titles = [
    'Lake Mead Near-Shore Wildfires - Obstacle-Aware vs Standard k-Means',
    'Lake Mead Near-Shore Wildfires - Cluster Summary',
    'Lake Mead Near-Shore Wildfires - Global Metrics',
    'Lake Mead Boundary',
]

if delete_existing:
    me = gis.users.me.username
    for title in item_titles:
        existing = gis.content.search(
            query=f'title:"{title}" owner:{me}',
            max_items=20,
        )
        existing = [it for it in existing if it.title == title]
        for it in existing:
            print(f'Deleting old item: {it.title} ({it.type}, ID {it.id})')
            it.delete()
    print('Cleanup complete.\n')
else:
    print('Skipping cleanup. New items will be created alongside any existing duplicates.\n')

Skipping cleanup. New items will be created alongside any existing duplicates.



### 5.1 Publish the Fires Feature Layer

In [8]:
# Item properties for the CSV (the resulting feature layer inherits these)
fire_item_properties = {
    'title': 'Lake Mead Near-Shore Wildfires - Obstacle-Aware vs Standard k-Means',
    'description': (
        'Wildfire occurrence data (1992-2020) within 5 km of Lake Mead\'s shoreline, '
        'clustered two ways for direct comparison: '
        '<b>standard k-Means</b> (geographic coordinates only) and '
        '<b>obstacle-aware k-Means</b> (geographic coordinates plus arc-length parameter '
        '<i>s</i> along the Lake Mead shoreline). Each fire has two cluster labels, '
        'one per method, so a dashboard can compare the two clusterings side by side.<br><br>'
        '<b>Source:</b> FPA FOD 6th Edition (USFS Research Data Archive).<br>'
        '<b>Method:</b> Obstacle-aware k-Means with optimized weight β selected '
        'from the J / span objective surface within the near-shore subset. See project '
        'repository for full methodology.'
    ),
    'tags': 'wildfires, clustering, Lake Mead, near-shore, obstacle-aware, k-means, portfolio',
    'type': 'CSV',
}

print('Uploading near-shore fire data to ArcGIS Online...')
fire_csv_item = gis.content.add(fire_item_properties, data=str(publish_path))
print(f'CSV uploaded: {fire_csv_item.title} (ID: {fire_csv_item.id})')

print('\nPublishing as hosted feature layer...')
fire_layer_item = fire_csv_item.publish()
print(f'Published: {fire_layer_item.title}')
print(f'Item URL: {fire_layer_item.homepage}')

Uploading near-shore fire data to ArcGIS Online...


c:\Users\mpp24\anaconda3\envs\obstacle-clustering\Lib\site-packages\IPython\core\interactiveshell.py:3701: DeprecatedWarning: add is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Folder.add()` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


CSV uploaded: Lake Mead Near-Shore Wildfires - Obstacle-Aware vs Standard k-Means (ID: d7f7a17279094dfda1eae00fe1ecad24)

Publishing as hosted feature layer...
Published: Lake Mead Near-Shore Wildfires - Obstacle-Aware vs Standard k-Means
Item URL: https://Michele-75.maps.arcgis.com/home/item.html?id=97d3c5271b304407beafae9fbf927d79


### 5.2 Publish the Supporting Tables

In [9]:
# Cluster summary table
summary_item_props = {
    'title': 'Lake Mead Near-Shore Wildfires - Cluster Summary',
    'description': (
        'Per-cluster summary statistics for the near-shore Lake Mead clustering, with '
        '4 rows for standard k-Means (method = "std") and 4 rows for obstacle-aware '
        'optimized (method = "oa_opt"). Includes n_fires, mean and median fire size, '
        'percent human-caused, and per-cluster arc-length span. Used to populate the '
        'dashboard side panels.'
    ),
    'tags': 'wildfires, clustering, Lake Mead, near-shore, summary, dashboard',
    'type': 'CSV',
}
summary_csv_path = processed_dir / 'mead_cluster_summary.csv'
summary_csv_item = gis.content.add(summary_item_props, data=str(summary_csv_path))
summary_table_item = summary_csv_item.publish()
print(f'Cluster summary published: {summary_table_item.title}')
print(f'  {summary_table_item.homepage}')

# Global metrics table
metrics_item_props = {
    'title': 'Lake Mead Near-Shore Wildfires - Global Metrics',
    'description': (
        'Headline metrics for the near-shore Lake Mead clustering. Single row with '
        'total fire count, year range, optimal β, mean arc-length span for both '
        'methods, and the percent span improvement. Used to populate the dashboard '
        'headline indicators.'
    ),
    'tags': 'wildfires, clustering, Lake Mead, near-shore, metrics, dashboard',
    'type': 'CSV',
}
metrics_csv_path = processed_dir / 'mead_global_metrics.csv'
metrics_csv_item = gis.content.add(metrics_item_props, data=str(metrics_csv_path))
metrics_table_item = metrics_csv_item.publish()
print(f'Global metrics published: {metrics_table_item.title}')
print(f'  {metrics_table_item.homepage}')

c:\Users\mpp24\anaconda3\envs\obstacle-clustering\Lib\site-packages\IPython\core\interactiveshell.py:3701: DeprecatedWarning: add is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Folder.add()` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Cluster summary published: Lake Mead Near-Shore Wildfires - Cluster Summary
  https://Michele-75.maps.arcgis.com/home/item.html?id=5d21a6ebaf004d4fa2ca1b51b1e2f31f


c:\Users\mpp24\anaconda3\envs\obstacle-clustering\Lib\site-packages\IPython\core\interactiveshell.py:3701: DeprecatedWarning: add is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Folder.add()` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Global metrics published: Lake Mead Near-Shore Wildfires - Global Metrics
  https://Michele-75.maps.arcgis.com/home/item.html?id=b79c2d1df31140219f7dd99a6dbcea87


### 5.3 Publish the Lake Mead Boundary Polygon

The boundary polygon doesn't change between runs, but having it as a hosted feature layer in AGOL makes it easy to add to either web map without re-uploading the GeoJSON every time. The cell converts the boundary CSV into a zipped shapefile (the most reliable format for AGOL polygon imports), uploads it, and publishes as a feature layer.

In [10]:
# Build a Shapely polygon from the saved boundary CSV
boundary_df = pd.read_csv(Path('../data/boundaries/mead_boundary.csv'))
coords = list(zip(boundary_df['longitude'], boundary_df['latitude']))
polygon = Polygon(coords)

gdf = gpd.GeoDataFrame(
    {'name': ['Lake Mead']},
    geometry=[polygon],
    crs='EPSG:4326'   # WGS 84
)

# Save to a shapefile (creates .shp + .dbf + .prj + .shx, all needed for AGOL)
shp_dir = Path('../data/boundaries/mead_boundary_shp')
shp_dir.mkdir(exist_ok=True)
gdf.to_file(shp_dir / 'mead_boundary.shp')

# Zip the shapefile directory for AGOL upload
zip_path = Path('../data/boundaries/mead_boundary')
shutil.make_archive(str(zip_path), 'zip', shp_dir)
zip_file = Path(f'{zip_path}.zip')
print(f'Saved zipped shapefile to {zip_file}')

# Upload as an AGOL item and publish as a hosted feature layer
boundary_item_props = {
    'title': 'Lake Mead Boundary',
    'description': (
        'Simplified shoreline polygon for Lake Mead, derived from the USGS National '
        'Hydrography Dataset (Waterbody layer 12). Built by unioning all NHD features '
        'above 1 sq km named "Lake Mead" to capture the main body plus the eastern '
        'arm, then simplifying with Douglas-Peucker (tolerance 0.005 degrees). Used '
        'as the boundary for obstacle-aware k-Means clustering and as a visual '
        'reference layer on the comparison dashboard.'
    ),
    'tags': 'Lake Mead, boundary, NHD, shoreline, portfolio',
    'type': 'Shapefile',
}

print('\nUploading boundary shapefile to ArcGIS Online...')
boundary_zip_item = gis.content.add(boundary_item_props, data=str(zip_file))
print(f'Shapefile uploaded: {boundary_zip_item.title} (ID: {boundary_zip_item.id})')

print('\nPublishing as hosted feature layer...')
boundary_layer_item = boundary_zip_item.publish()
print(f'Published: {boundary_layer_item.title}')
print(f'Item URL: {boundary_layer_item.homepage}')

Saved zipped shapefile to ..\data\boundaries\mead_boundary.zip

Uploading boundary shapefile to ArcGIS Online...


c:\Users\mpp24\anaconda3\envs\obstacle-clustering\Lib\site-packages\IPython\core\interactiveshell.py:3701: DeprecatedWarning: add is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Folder.add()` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Shapefile uploaded: Lake Mead Boundary (ID: 75f51941c7c144d8979b18cc8a454b4b)

Publishing as hosted feature layer...
Published: Lake Mead Boundary
Item URL: https://Michele-75.maps.arcgis.com/home/item.html?id=56c69e52d8d148ac9764a53e286f9cd5


### 5.4 Share Publicly

In [11]:
for item in [fire_layer_item, summary_table_item, metrics_table_item, boundary_layer_item]:
    item.share(everyone=True)
    print(f'Shared: {item.title}')

print()
print('Item URLs (copy these for the dashboard build below):')
print(f'  Feature layer:   {fire_layer_item.homepage}')
print(f'  Cluster summary: {summary_table_item.homepage}')
print(f'  Global metrics:  {metrics_table_item.homepage}')
print(f'  Boundary layer:  {boundary_layer_item.homepage}')

c:\Users\mpp24\anaconda3\envs\obstacle-clustering\Lib\site-packages\IPython\core\interactiveshell.py:3701: DeprecatedWarning: share is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Item.sharing` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Shared: Lake Mead Near-Shore Wildfires - Obstacle-Aware vs Standard k-Means


c:\Users\mpp24\anaconda3\envs\obstacle-clustering\Lib\site-packages\IPython\core\interactiveshell.py:3701: DeprecatedWarning: share is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Item.sharing` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Shared: Lake Mead Near-Shore Wildfires - Cluster Summary


c:\Users\mpp24\anaconda3\envs\obstacle-clustering\Lib\site-packages\IPython\core\interactiveshell.py:3701: DeprecatedWarning: share is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Item.sharing` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Shared: Lake Mead Near-Shore Wildfires - Global Metrics


c:\Users\mpp24\anaconda3\envs\obstacle-clustering\Lib\site-packages\IPython\core\interactiveshell.py:3701: DeprecatedWarning: share is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Item.sharing` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Shared: Lake Mead Boundary

Item URLs (copy these for the dashboard build below):
  Feature layer:   https://Michele-75.maps.arcgis.com/home/item.html?id=97d3c5271b304407beafae9fbf927d79
  Cluster summary: https://Michele-75.maps.arcgis.com/home/item.html?id=5d21a6ebaf004d4fa2ca1b51b1e2f31f
  Global metrics:  https://Michele-75.maps.arcgis.com/home/item.html?id=b79c2d1df31140219f7dd99a6dbcea87
  Boundary layer:  https://Michele-75.maps.arcgis.com/home/item.html?id=56c69e52d8d148ac9764a53e286f9cd5


## 6. Building the Standard $k$-Means Web Map

This is the first of two web maps. Both use the same feature layer (the near-shore fires we just published) along with your Lake Mead boundary polygon layer. Each web map styles the points by a different cluster column: standard map by `Cluster_Std`, obstacle-aware map by `Cluster_OA`.

### Step 1: Open the layer in the Map Viewer

1. Go to the feature layer's item page (printed above as "Feature layer").
2. Click **Open in Map Viewer** in the top right.
3. The layer loads with default symbology.

### Step 2: Add the Lake Mead boundary layer

1. Click **Add** in the left rail.
2. Switch the search to **My content**.
3. Find your `Lake Mead Boundary` layer (uploaded earlier) and add it.
4. In the **Layers** panel, drag the boundary layer **below** the fires layer so the points stay visible on top.
5. Click the boundary layer, then **Styles** in the right rail. Set:
   - **Fill**: transparent (or very light blue with high transparency)
   - **Outline color**: `#2c3e50` (slate)
   - **Outline width**: 1.5-2 px

### Step 3: Style the clusters by `Cluster_Std`

1. With the fires layer selected, click **Styles** in the right rail.
2. **Choose attributes**: click **+ Field** and pick `Cluster_Std`. Auto-detected as a category field.
3. **Pick a style**: choose **Types (unique symbols)** and click **Style options**.
4. Click each cluster's symbol to set its color from the project palette:
   - Cluster 1 -> `#e74c3c` (red)
   - Cluster 2 -> `#3498db` (blue)
   - Cluster 3 -> `#2ecc71` (green)
   - Cluster 4 -> `#f39c12` (orange)
5. For each cluster symbol, set:
   - **Size**: 10 px (slightly larger than basin maps since we have fewer points)
   - **Outline color**: dark grey (`#282828`)
   - **Outline width**: 0.5 px
   - **Outline transparency**: ~30%
   - **Transparency**: ~20%
6. Click **Done** on each panel.

### Step 4: Configure the popup

1. With the fires layer selected, click **Pop-ups** and toggle popups on.
2. **Title**: `Fire in Cluster {Cluster_Std} (standard k-means)`.
3. **Fields list** -- show:
   - Cluster_Std (label: "Standard Cluster")
   - Cluster_OA (label: "Obstacle-Aware Cluster")
   - Cause
   - Specific_Cause (label: "Specific Cause")
   - Fire_Size_Acres (label: "Fire Size (acres)", 2 decimal places)
   - Fire_Year (label: "Year")
   - Dist_To_Lake_Km (label: "Distance to Lake (km)", 2 decimal places)
4. Hide LATITUDE, LONGITUDE, and any auto-added ObjectID fields.

### Step 5: Set the basemap and extent

1. Click **Basemap** in the left rail. Pick **Topographic**.
2. Pan/zoom so Lake Mead and its near-shore fires fill the view -- you can zoom in tighter than the basin view since we're only showing 214 fires. Aim for the lake's full outline to be visible with a comfortable margin.
3. Click **Save and open** -> **Save as**.

### Step 6: Save the web map

1. **Title**: `Lake Mead Near-Shore Wildfires - Standard k-Means Web Map`
2. **Tags**: `wildfires, clustering, Lake Mead, near-shore, k-means, portfolio`
3. **Summary**: `Lake Mead wildfires within 5 km of shore (1992-2020) clustered with standard k-means.`
4. Click **Save**.

### Step 7: Share publicly

1. **Share** -> **Everyone (public)**.
2. Copy the URL into the cell below.

In [ ]:
std_webmap_url = ''
print(f'Standard k-Means web map URL: {std_webmap_url}')

## 7. Building the Obstacle-Aware Web Map

Same recipe as the standard map but colors by `Cluster_OA`.

### Step 1-3: Same layer, different style field

1. Open the same feature layer in the Map Viewer (or use **Save as** from the standard k-means map and edit).
2. Re-add the Lake Mead boundary layer if needed (it should carry over from Save as).
3. **Styles** -> **+ Field** -> pick `Cluster_OA` (not `Cluster_Std`).
4. Apply the same Types (unique symbols) styling with the same four colors and symbol settings as the standard map.

### Step 4: Configure the popup

Same fields visible as the standard map, but update the popup title:

1. **Title**: `Fire in Cluster {Cluster_OA} (obstacle-aware)`
2. Same field list (showing both Cluster_Std and Cluster_OA so viewers can see how the same fire is labeled by both methods).

### Step 5-6: Same basemap and extent, save

1. Use the same basemap and extent as the standard k-means map so the side-by-side comparison reads cleanly.
2. Save as: `Lake Mead Near-Shore Wildfires - Obstacle-Aware Web Map`
3. **Tags**: `wildfires, clustering, Lake Mead, near-shore, obstacle-aware, portfolio`
4. **Summary**: `Lake Mead wildfires within 5 km of shore (1992-2020) clustered with obstacle-aware k-means.`
5. Click **Save**.

### Step 7: Share publicly

In [ ]:
oa_webmap_url = ''
print(f'Obstacle-Aware web map URL: {oa_webmap_url}')

## 8. Building the Comparison Dashboard

The dashboard puts both web maps side by side. With only 214 fires per map, individual fire reassignments between methods are easier to spot than they were at the basin scale.

### Target layout

```
+------------------------------------------------------------+
|  Lake Mead Near-Shore Wildfires --                         |
|  Standard vs Obstacle-Aware Clustering                     |
|  Within 5 km of the lake, the obstacle parameter improves  |
|  arc-length span by ~19%.                                  |
+----------------+---------------------+---------------------+
|   Standard     |     Obstacle-Aware  |                     |
|   k-Means      |     k-Means         |  Span improvement:  |
|                |                     |     19%             |
|     [MAP]      |       [MAP]         |                     |
|                |                     |  Near-shore fires:  |
|                |                     |     214             |
| Cluster Legend | Cluster Legend      |                     |
|                |                     |  Cause: [pie chart] |
| Mean Fire Size | Mean Fire Size      |                     |
+----------------+---------------------+---------------------+
| Source: FPA FOD 6th Edition (USFS) | Within 5 km of shore  |
+------------------------------------------------------------+
```

### Step 1: Create the dashboard

1. Open `https://michele-75.maps.arcgis.com/apps/dashboards/` and click **New dashboard**.
2. Title it **Lake Mead Near-Shore Wildfires - Standard vs Obstacle-Aware Clustering** and click **Create dashboard**.

### Step 2: Add the two maps

1. In the dashboard editor, click **+** (top left) -> **Map**.
2. Pick **Lake Mead Near-Shore Wildfires - Standard k-Means Web Map** for the left map.
3. Enable **Pop-ups** and **Legend** in the configuration panel.
4. Click **Done**.
5. Repeat for the right side with **Lake Mead Near-Shore Wildfires - Obstacle-Aware Web Map**.
6. Drag the second map to the right of the first, leaving room on the right side for the indicators.

### Step 3: Add the headline indicator (span improvement)

This is the dashboard's main story.

1. Click **+** -> **Indicator**.
2. Data source: **Lake Mead Near-Shore Wildfires - Global Metrics**.
3. No filter needed (only one row).
4. **Indicator** tab:
   - **Value**: `span_improvement_pct`
   - **Statistic**: `Sum`
   - **Decimal places**: 0 (or 1 if you prefer)
5. **General** tab:
   - **Top text**: `Obstacle-aware improvement`
   - **Middle text suffix**: `%`
   - **Bottom text**: `over standard k-means`
   - **Background color**: `#f59e0b` (amber)
   - **Text color**: `#1a1a1a`
6. Click **Done**. Position at the top of the right column.

### Step 4: Add the near-shore fires indicator

1. Click **+** -> **Indicator**.
2. Data source: **Lake Mead Near-Shore Wildfires - Global Metrics**.
3. **Value**: `total_fires`. **Statistic**: `Sum`. **Decimal places**: 0.
4. **Top text**: `Near-shore fires (1992-2020)`.
5. **Bottom text**: `within 5 km of shore`.
6. Click **Done**. Position below the headline indicator.

### Step 5: Add the cluster legend lists

Each list shows the four clusters for its map and filters that map's view when clicked.

**Left list (for the standard map)**:

1. Click **+** -> **List**.
2. Data source: **Lake Mead Near-Shore Wildfires - Cluster Summary**.
3. **Filter** tab: `method is std`.
4. **List** tab -> **Line item template**:
   ```
   <b>Cluster {cluster_label}</b><br>
   n = {n_fires} fires &nbsp;|&nbsp; {pct_human}% human-caused
   ```
   **Maximum results**: 10.
5. **General** tab: **Title** = `Standard k-means clusters`.
6. **Actions** tab: in the **Filter** section, target the standard $k$-means map. Source field `cluster_label`, target field `Cluster_Std`.
7. Click **Done**. Position below the standard map.

**Right list (for the obstacle-aware map)**:

1. Same as above but **Filter** = `method is oa_opt`.
2. **Title** = `Obstacle-aware clusters`.
3. **Actions** tab: target the obstacle-aware map. Source field `cluster_label`, target field `Cluster_OA`.
4. Position below the obstacle-aware map.

### Step 6: Add the per-cluster mean fire size indicators

**Left indicator (standard method)**:

1. Click **+** -> **Indicator**.
2. Data source: **Lake Mead Near-Shore Wildfires - Cluster Summary**.
3. **Filter** tab: `method is std`.
4. **Indicator** tab:
   - **Value**: `mean_fire_size`
   - **Statistic**: `Mean`
   - **Decimal places**: 1
5. **General** tab:
   - **Top text**: `Mean fire size`
   - **Middle text suffix**: ` acres`
6. **Actions** tab: enable this indicator as a filter target from the standard $k$-means legend list.
7. Click **Done**. Position below the standard legend list.

**Right indicator (obstacle-aware method)**: same as above but `method is oa_opt`, and enable from the obstacle-aware legend list. Position below the obstacle-aware legend list.

### Step 7: Add the fire cause pie chart

1. Click **+** -> **Pie chart**.
2. Data source: **Lake Mead Near-Shore Wildfires - Obstacle-Aware vs Standard k-Means** (the fires feature layer).
3. **Categories from**: **Grouped values** -> **Cause**.
4. **Sort by**: **Category** -> **Ascending**.
5. **General** tab:
   - **Title**: `Cause`
   - Custom colors:
     - Human-Caused: `#3a3a3a`
     - Natural: `#8b6f47`
6. Click **Done**. Position in the right column under the fire count indicator.

### Step 8: Style the dashboard

1. **Page header**:
   - **Background color**: `#2c3e50` (slate)
   - **Text color**: `#f4f1ec` (warm off-white)
   - **Title**: `Lake Mead Near-Shore Wildfires - Standard vs Obstacle-Aware Clustering`
   - **Subtitle**: `Within 5 km of the lake, the obstacle parameter` *s* `improves arc-length span by ~19% compared to standard k-means.`
2. **Page background**: `#f4f1ec`
3. **Indicator element backgrounds**: `#faf7f1` (warm off-white) for all non-headline indicators
4. **List element backgrounds**: same `#faf7f1`
5. **Body text color**: `#2c3e50`

### Step 9: Footer

1. Click **+** -> **Rich text**. Position at the bottom.
2. Content:
   ```
   Wildfire data: FPA FOD 6th Edition (USFS), 1992-2020, fires with known causes only, within 5 km of Lake Mead shoreline
   | Method: obstacle-aware k-means | Author: Michele Perry | [GitHub repository link]
   ```
3. Left-align, 11-12 px, color `#6b7280`.

### Step 10: Save and share

1. Click **Save** in the dashboard toolbar.
2. From the dashboard's item page, set sharing to **Everyone (public)**.
3. Copy the URL into the cell below.

In [ ]:
dashboard_url = ''
print(f'Dashboard URL: {dashboard_url}')

## 9. Updating Items In Place (Optional)

If you re-run Notebook 03 and produce new CSVs (different threshold, different β, etc.), use this Update Data flow to refresh the AGOL items without breaking the dashboard's references to them. Item IDs and sharing settings are preserved.

In [ ]:
def update_existing_item(title, local_csv_path):
    """Find an existing item by title and replace its data."""
    me = gis.users.me.username
    matches = gis.content.search(
        query=f'title:"{title}" owner:{me}',
        max_items=20,
    )
    matches = [it for it in matches if it.title == title]

    if not matches:
        print(f'  No item found with title "{title}". Skipping update.')
        return None

    if len(matches) > 1:
        print(f'  Multiple items found with title "{title}". '
              f'Updating the first one ({matches[0].id}).')

    item = matches[0]
    print(f'  Updating item: {item.title} (ID {item.id})')
    item.update(data=str(local_csv_path))
    return item


# Uncomment to perform the update
# print('Updating fires feature layer source CSV...')
# update_existing_item(
#     'Lake Mead Near-Shore Wildfires - Obstacle-Aware vs Standard k-Means',
#     publish_path,
# )

# print('\nUpdating cluster summary table...')
# update_existing_item(
#     'Lake Mead Near-Shore Wildfires - Cluster Summary',
#     processed_dir / 'mead_cluster_summary.csv',
# )

# print('\nUpdating global metrics table...')
# update_existing_item(
#     'Lake Mead Near-Shore Wildfires - Global Metrics',
#     processed_dir / 'mead_global_metrics.csv',
# )

## 10. Summary

What this notebook accomplished:

1. Loaded the three CSVs from Notebook 03 (now configured for the near-shore subset)
2. Filtered the fires to those within 5 km of Lake Mead, prepared friendly columns
3. Uploaded the three CSVs to ArcGIS Online and published them as hosted items
4. Provided step-by-step instructions for building the two web maps and the comparison dashboard
5. Included an optional Update Data flow for re-publishing without breaking the dashboard

The dashboard's story: 214 wildfires within 5 km of Lake Mead, clustered two ways. Standard $k$-means produces clusters that don't account for the water; obstacle-aware $k$-means produces clusters that do, and tightens arc-length span by ~19% in the process. With this smaller subset, individual fire reassignments between methods are easy to see, making the visual comparison more impactful than the basin-scale view.

Lake Tahoe's results (Notebook 02) and the full Mead basin results (Notebook 03) provide context for the writeup, but the dashboard focuses on this near-shore Mead comparison for visual clarity.